# Cisco CX CDC: Ibis / Snowflake Backend (Portable)

**Ibis DataFrame API with Snowflake as the compute backend** — no DuckDB.

- **Cloud (SPCS):** `ibis.snowflake.from_connection(session)` — CDC runs on Snowflake warehouse
- **On-Prem (EC2/Docker):** `ibis.snowflake.from_connection(sf_conn)` — CDC runs on Snowflake warehouse via connector

The Ibis expressions compile to Snowflake SQL and execute on the warehouse. Same API, same results, either platform.

In [1]:
import subprocess, importlib, sys
required = [('ibis-framework[snowflake]', 'ibis'), ('pyarrow', 'pyarrow'), ('snowflake-connector-python[pandas]', 'snowflake.connector')]
for pkg, mod in required:
    try:
        importlib.import_module(mod.split('.')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages ready')

All packages ready


In [2]:
import os, time
import ibis
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs

TARGET_DB = 'CISCO_CX_PILOT'
TARGET_SCHEMA = 'PROCESSED'
PREV_TABLE = 'PREV_SNAPSHOT_STAGING'
RESULT_TABLE = 'IBIS_SF_CDC_RESULT'
S3_BUCKET = 'cisco-cx-cdc-pilot'

COMPARE_COLS = ['SOFTWARE_VERSION', 'CPU_UTILIZATION', 'MEMORY_UTILIZATION',
                'CRITICAL_BUGS_COUNT', 'CONTRACT_STATUS', 'IP_ADDRESS']

print(f'Target: {TARGET_DB}.{TARGET_SCHEMA}.{RESULT_TABLE}')
print(f'S3: s3://{S3_BUCKET}/cdc_data/')

Target: CISCO_CX_PILOT.PROCESSED.IBIS_SF_CDC_RESULT
S3: s3://cisco-cx-cdc-pilot/cdc_data/


In [3]:
t0 = time.time()

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    PLATFORM = 'SPCS'
    sf_raw = session.connection
    con = ibis.snowflake.from_connection(sf_raw, create_object_udfs=False)
    print(f'Platform: Snowflake SPCS (from_connection)')
except Exception:
    import snowflake.connector
    conn_name = os.getenv('SNOWFLAKE_CONNECTION_NAME', 'default')
    sf_raw = snowflake.connector.connect(
        connection_name=conn_name,
        database=TARGET_DB,
        schema=TARGET_SCHEMA,
        warehouse='HOL_ICE_WH'
    )
    PLATFORM = 'EC2'
    con = ibis.snowflake.from_connection(sf_raw, create_object_udfs=False)
    print(f'Platform: EC2/On-Prem (connector: {conn_name})')

t_connect = time.time() - t0
print(f'  Ibis Snowflake backend connected in {t_connect:.1f}s')
print(f'  Ibis version: {ibis.__version__}')

Platform: EC2/On-Prem (connector: default)
  Ibis Snowflake backend connected in 7.6s
  Ibis version: 12.0.0


In [4]:
t1 = time.time()

prev_tbl = con.table(PREV_TABLE, database=(TARGET_DB, TARGET_SCHEMA))
prev_count = prev_tbl.count().execute()
t_read_prev = time.time() - t1
print(f'  prev_snapshot from Snowflake table: {prev_count:,} rows (ref in {t_read_prev:.1f}s)')

  prev_snapshot from Snowflake table: 10,000,000 rows (ref in 0.3s)


In [5]:
t2 = time.time()
t_read_curr = 0.0
print('  curr_snapshot: will be loaded server-side via COPY INTO from external stage (no client transfer)')

  curr_snapshot: will be loaded server-side via COPY INTO from external stage (no client transfer)


In [6]:
t3 = time.time()

sf_raw.cursor().execute(f'CREATE TABLE IF NOT EXISTS {TARGET_DB}.{TARGET_SCHEMA}.IBIS_CDC_CURR_STAGING LIKE {TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}')
sf_raw.cursor().execute(f'TRUNCATE TABLE {TARGET_DB}.{TARGET_SCHEMA}.IBIS_CDC_CURR_STAGING')

sf_raw.cursor().execute(f"""
COPY INTO {TARGET_DB}.{TARGET_SCHEMA}.IBIS_CDC_CURR_STAGING
FROM @{TARGET_DB}.LANDING_ZONE.CDC_S3_STAGE/curr_snapshot/
FILE_FORMAT = (TYPE = 'PARQUET')
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
FORCE = TRUE
""")

curr_tbl = con.table('IBIS_CDC_CURR_STAGING', database=(TARGET_DB, TARGET_SCHEMA))
curr_count = curr_tbl.count().execute()
t_load_curr = time.time() - t3
print(f'  curr loaded to Snowflake via COPY INTO (server-side S3 read): {curr_count:,} rows in {t_load_curr:.1f}s')

  curr loaded to Snowflake via COPY INTO (server-side S3 read): 10,020,000 rows in 9.0s


In [7]:
t4 = time.time()

compare_expr_p = ibis.literal('|').join([prev_tbl[c].cast('string') for c in COMPARE_COLS])
prev_hashed = prev_tbl.mutate(_hash_prev=compare_expr_p.hash())

compare_expr_c = ibis.literal('|').join([curr_tbl[c].cast('string') for c in COMPARE_COLS])
curr_hashed = curr_tbl.mutate(_hash_curr=compare_expr_c.hash())

prev_keys = prev_hashed.select('DEVICE_ID', '_hash_prev')
curr_keys = curr_hashed.select('DEVICE_ID', '_hash_curr')

merged = curr_keys.outer_join(prev_keys, 'DEVICE_ID')

n_inserts = merged.filter(merged._hash_prev.isnull()).count().execute()
n_deletes = merged.filter(merged._hash_curr.isnull()).count().execute()
both = merged.filter(merged._hash_curr.notnull() & merged._hash_prev.notnull())
n_updates = both.filter(both._hash_curr != both._hash_prev).count().execute()

t_cdc = time.time() - t4
print(f'  CDC computed on Snowflake warehouse in {t_cdc:.1f}s')
print(f'  Inserts: {n_inserts:,}')
print(f'  Updates: {n_updates:,}')
print(f'  Deletes: {n_deletes:,}')
print(f'  Total changes: {n_inserts + n_updates + n_deletes:,}')

  CDC computed on Snowflake warehouse in 5.1s
  Inserts: 25,000
  Updates: 18,952
  Deletes: 5,000
  Total changes: 48,952


In [8]:
t5 = time.time()

insert_ids_pdf = merged.filter(merged._hash_prev.isnull()).select('DEVICE_ID').execute()
update_ids_pdf = both.filter(both._hash_curr != both._hash_prev).select('DEVICE_ID').execute()
delete_ids_pdf = merged.filter(merged._hash_curr.isnull()).select('DEVICE_ID').execute()

insert_set = set(insert_ids_pdf['DEVICE_ID'])
update_set = set(update_ids_pdf['DEVICE_ID'])
delete_set = set(delete_ids_pdf['DEVICE_ID'])
all_upsert_set = insert_set | update_set

print(f'  IDs extracted: {len(insert_set):,} inserts, {len(update_set):,} updates, {len(delete_set):,} deletes')

all_upsert_list = list(all_upsert_set)
upsert_expr = curr_tbl.filter(curr_tbl.DEVICE_ID.isin(all_upsert_list))
upsert_pdf = upsert_expr.execute()
upsert_pdf['CDC_ACTION'] = upsert_pdf['DEVICE_ID'].apply(lambda x: 'INSERT' if x in insert_set else 'UPDATE')

delete_list = list(delete_set)
delete_expr = prev_tbl.filter(prev_tbl.DEVICE_ID.isin(delete_list))
delete_pdf = delete_expr.execute()
delete_pdf['CDC_ACTION'] = 'DELETE'

result_pdf = pd.concat([upsert_pdf, delete_pdf], ignore_index=True)

from snowflake.connector.pandas_tools import write_pandas
sf_raw.cursor().execute(f'''CREATE TABLE IF NOT EXISTS {TARGET_DB}.{TARGET_SCHEMA}.{RESULT_TABLE} (
    DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, HOSTNAME VARCHAR,
    SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT,
    CRITICAL_BUGS_COUNT NUMBER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR,
    LAST_SEEN TIMESTAMP_NTZ, CDC_ACTION VARCHAR
)''')
sf_raw.cursor().execute(f'TRUNCATE TABLE {TARGET_DB}.{TARGET_SCHEMA}.{RESULT_TABLE}')
write_pandas(sf_raw, result_pdf, RESULT_TABLE, database=TARGET_DB, schema=TARGET_SCHEMA)

t_write = time.time() - t5
print(f'  CDC results written to {RESULT_TABLE}: {len(result_pdf):,} rows in {t_write:.1f}s')

  IDs extracted: 25,000 inserts, 18,952 updates, 1 deletes


  CDC results written to IBIS_SF_CDC_RESULT: 43,952 rows in 23.3s


In [9]:
total = time.time() - t0
print('\n' + '='*60)
print(f'IBIS/SNOWFLAKE CDC PIPELINE COMPLETE ({PLATFORM})')
print('='*60)
print(f'  Connect:                    {t_connect:.1f}s')
print(f'  Read prev (SF table ref):   {t_read_prev:.1f}s')
print(f'  Load curr (COPY INTO S3):   {t_load_curr:.1f}s')
print(f'  CDC compute (SF WH):        {t_cdc:.1f}s')
print(f'  Extract + Write results:    {t_write:.1f}s')
print(f'  TOTAL:                      {total:.1f}s')
print('='*60)
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')
print(f'  Total rows written: {len(result_pdf):,}')


IBIS/SNOWFLAKE CDC PIPELINE COMPLETE (EC2)
  Connect:                    7.6s
  Read prev (SF table ref):   0.3s
  Load curr (COPY INTO S3):   9.0s
  CDC compute (SF WH):        5.1s
  Extract + Write results:    23.3s
  TOTAL:                      45.4s
  Inserts: 25,000 | Updates: 18,952 | Deletes: 5,000
  Total rows written: 43,952


In [10]:
sf_raw.cursor().execute(f'DROP TABLE IF EXISTS {TARGET_DB}.{TARGET_SCHEMA}.IBIS_CDC_CURR_STAGING')
print('Cleanup: dropped IBIS_CDC_CURR_STAGING')

Cleanup: dropped IBIS_CDC_CURR_STAGING
